# Katip Mask Nano v2 · same 2.5 MB size, taught by Katip Mask Base

**What this does.** Retrains Nano (2 layers, 128 hidden, 16k vocabulary, ~2.6 MB int8) with Base as its teacher, on
real text (Wiki NER train split, court decisions with refilled names, foreign-name sets) plus Katip's synthetic
documents. The vocabulary is learned on that real text; Nano v1.1's was learned on templates only, a known cause of its misses.

**Honest expectation:** a few points on general prose, not a leap. Under 5 MB we have never passed ~55% on Wikipedia
names; that is why Mini exists. Nano stays the phone / slow-connection option.

| | |
|---|---|
| Time | ~5 min smoke test, then ~30-45 min on a free T4 |
| Output | `katip-mask-nano-v2.zip` and a results table (compare: Nano v1.1 ~51% Wikipedia recall) |

**How to run:** Runtime > Run all.

In [ ]:
# 1 · Check the GPU, get the code. Run this cell first. (Takes ~1 min.)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE. In Colab: Runtime > Change runtime type > T4 GPU. In Kaggle: Settings > Accelerator > GPU T4.")
if not os.path.exists("katip"):
    subprocess.run("git clone -q --depth 1 -b legal-study https://github.com/fikrifirat/katip", shell=True, check=True)
os.chdir("katip")
subprocess.run(f"{sys.executable} -m pip install -q regex tokenizers onnx onnxruntime faker huggingface_hub 'transformers>=4.44' accelerate", shell=True, check=True)
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Optional · Hugging Face token, only needed to upload at the end.
# Colab: the key icon on the left > add a secret named HF_TOKEN (write access) and allow this notebook.
# Kaggle: Add-ons > Secrets > HF_TOKEN. Nothing is hard-coded here.
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token:", "found" if HF_TOKEN else "none (fine: you can download the zip instead)")

## 2 · Smoke test (5 min)

In [ ]:
!cd lab && MODE=smoke STUDENTS=tiny VOCAB=16000 LR=1e-3 NAME=katip-mask-nano-v2 python mini/train_mini.py 2>&1 | grep -v Warning | tail -5

## 3 · Full run (~40 min)

In [ ]:
!cd lab && MODE=full STUDENTS=tiny VOCAB=16000 LR=1e-3 NAME=katip-mask-nano-v2 python mini/train_mini.py 2>&1 | grep -v Warning | tail -20

In [ ]:
import json
r = json.load(open("lab/mini/runs/katip-mask-nano-v2_results.json"))
w = r["winner"]; t = r["test"]["wiki test"]
print(f"Nano v2: {w['int8_mb']} MB  trust {w['trust']}  Wikipedia recall {t['recall']:.1%}  precision {t['precision']:.1%}  (v1.1: ~51%)")
print(f"blind {r['test']['blind']['recall']:.1%}  false names on name-free text {r['test']['negatives']['false']}")

In [ ]:
# Save the result: a zip you can download (left panel > Files) and, if you set HF_TOKEN, a PRIVATE upload to Hugging Face.
# Make it public later on the model page (Settings > Visibility) after checking the scores.
import shutil, os
z = shutil.make_archive("katip-mask-nano-v2", "zip", "lab/mini/runs")
print("zip:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(z)
except Exception:
    pass
UPLOAD = False   # set True to upload
if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo("f1k/katip-mask-nano-v2", private=True, exist_ok=True)
    api.upload_folder(folder_path="lab/mini/runs", repo_id="f1k/katip-mask-nano-v2", commit_message="trained in a notebook (katip-mask-nano-v2)")
    print("uploaded to https://huggingface.co/f1k/katip-mask-nano-v2 (private)")